# LLM-as-a-Judge v2: Contextual Attachment Scoring

This notebook tests the **improved scoring approach** that addresses the sparse high-attachment distribution (v1 had only 1.3% with scores ≥5).

## Key Improvements in v2

1. **Contextual Attachment Scoring**: Judge sees BOTH assistant response AND user reply (not just user reply in isolation)
2. **Single-Pass Strict Scoring**: Uses the base contextual rubric only (no lenient aggregation)
3. **Cleaner Implementation**: Simplified from dual-pass to single contextual evaluation

## Scoring Scheme

- **Empathy Score (T):** 1-7, applied to `llm_response`
- **Attachment Score (Y):** 1-7, applied to `user_reply` WITH `llm_response` context

## Judge Model

Using **Llama 3.1 8B Instant** via Groq API

In [1]:
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import time
from groq import Groq

# Add scripts directory to path
sys.path.append('../scripts')

# Load environment variables
load_dotenv()

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [2]:
# Test API connection
try:
    groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": "Hello, respond with just 'OK'"}],
        max_tokens=5
    )
    print("✅ Groq/Llama API Success!")
    print(f"Response: {response.choices[0].message.content}")
except Exception as e:
    print(f"❌ Groq/Llama API Error: {e}")
    print("\nMake sure you have set GROQ_API_KEY in your .env file")

✅ Groq/Llama API Success!
Response: OK


## 1. Load Sample Dataset

In [3]:
# Load the sample preprocessed dataset
sample_path = '../data/filtered/wildchat_sample_preprocessed.csv'

print(f"Loading sample dataset from: {sample_path}")
df_sample = pd.read_csv(sample_path)

print(f"\nDataset shape: {df_sample.shape}")
print(f"\nColumns: {list(df_sample.columns)}")
print(f"\nFirst few rows:")
df_sample.head()

Loading sample dataset from: ../data/filtered/wildchat_sample_preprocessed.csv

Dataset shape: (223, 14)

Columns: ['conversation_hash', 'turn_pair_id', 'model', 'user_id', 'turn_number', 'total_turns', 'timestamp', 'hour_of_day', 'user_prompt', 'llm_response', 'user_reply', 'turn_identifier', 'hashed_ip', 'country']

First few rows:


,conversation_hash,turn_pair_id,model,user_id,turn_number,total_turns,timestamp,hour_of_day,user_prompt,llm_response,user_reply,turn_identifier,hashed_ip,country
0,0d2231ad2232cb26b6b4fb75a19ddc93,0d2231ad2232cb26b6b4fb75a19ddc93_1701994,gpt-3.5-turbo-0613,95f425b87fe2e14b52175f1575429e26,5,14,2023-10-27 05:58:57+00:00,5,what is the archetype of the bluest eye,"The ""The Bluest Eye"" by Toni Morrison incorporates various archetypes throughout the narrative to explore different themes. One significant archetype present in the novel is the journey or quest a...",explore the archetypes in this extract in the Bluest Eye:\n,1701994,eb89e19f722505ebc824d899129abab71e3cfc6da9102b8501b633c9461895a4,Cambodia
1,350c2f00ead0f1414ef4edf68f352d9e,350c2f00ead0f1414ef4edf68f352d9e_1697823,gpt-3.5-turbo-0613,35356662f3438ac79d366ccdbf648334,1,16,2023-10-25 19:51:32+00:00,19,hey,Hello! How can I assist you today?,I'm going to show you working thesis proposal so far okay,1697823,379b15a02573b91046686505cc03aa490a04b2c5117abb98efaa2d62b6578d02,United States
2,1f8e543b0a918d6334270ec1e6860943,1f8e543b0a918d6334270ec1e6860943_532466,gpt-3.5-turbo-0301,2a7d71b2f8f1232d62fbc922abed5de2,3,6,2023-05-18 14:19:10+00:00,14,use the web,"Using the web, I found a push-up test called the Army Physical Fitness Test (APFT). According to the APFT standards, a 17-21-year-old female weighing 96 pounds should be able to do at least 29 pus...",how many bodyweight squats should a 100 pound be able to do before failure?,532466,8bf0c535828398561144dadbe28201281ec5338f103c6631868c4456e9ae069d,United States
3,8e24f7f88936429c0ce951519990342b,8e24f7f88936429c0ce951519990342b_2781437,gpt-4-0125-preview,f09dbc1215f47441faba8e1551969148,1,6,2024-04-25 07:15:49+00:00,7,"short it ""Unveil the secrets of “Dinosaur DNA: The Genetic Time Travel,” where cutting-edge science meets the ethical dilemmas of reviving the extinct. Join Dr. Eleanor Hayes in a groundbreaking a...","""Discover the intriguing blend of science and ethics in ""Dinosaur DNA: The Genetic Time Travel."" Follow Dr. Eleanor Hayes on a revolutionary journey in genetics to resurrect an Apatosaurus, facing...",short it,2781437,1ce047da99f5a37579066b48aab8335f4702af8e3ca83825ac1b7716017cff6e,India
4,8e24f7f88936429c0ce951519990342b,8e24f7f88936429c0ce951519990342b_2781442,gpt-4-0125-preview,f09dbc1215f47441faba8e1551969148,3,6,2024-04-25 07:15:49+00:00,7,short it,"""Explore the fusion of science and ethics in 'Dinosaur DNA: The Genetic Time Travel.' Join Dr. Eleanor Hayes on her quest to revive an Apatosaurus, navigating groundbreaking genetics and moral dil...","short it ""“Dive into ‘The Solar Sailors: Exploring the Power of the Sun,’ a thrilling voyage where innovation meets the cosmos. Join Dr. Aria Kim’s crew on the Helios as they sail the stars, power...",2781442,1ce047da99f5a37579066b48aab8335f4702af8e3ca83825ac1b7716017cff6e,India


## 2. Import Scoring Function (v2)

This uses `score_conversations2.py` which implements:
- Contextual attachment scoring (assistant response + user reply)
- Strict single-pass evaluation
- Direct assignment (no aggregation)

In [7]:
# Import the contextual scoring function
from score_conversations2 import score_conversations

print("✓ Imported scoring function from score_conversations2.py")
print("\nThis version:")
print("  - Uses BOTH assistant response + user reply for attachment scoring (contextual)")
print("  - Single-pass strict evaluation (no lenient aggregation)")
print("  - Attachment score = strict contextual score (1-7)")

✓ Imported scoring function from score_conversations2.py

This version:
  - Uses BOTH assistant response + user reply for attachment scoring (contextual)
  - Single-pass strict evaluation (no lenient aggregation)
  - Attachment score = strict contextual score (1-7)


## 3. Score a Small Subset (5 pairs)

Test the new approach on 5 turn pairs first

In [8]:
# Score a small subset first (5 turn pairs) to test
print("Scoring a small subset (5 turn pairs) with v2 approach...")
print("="*70)

df_subset = df_sample.head(5).copy()
df_subset_scored = score_conversations(df_subset, verbose=True)

Scoring a small subset (5 turn pairs) with v2 approach...
--- Processing turn pair 1/5 (ID: 0d2231ad2232cb26b6b4fb75a19ddc93_1701994) ---
  Getting empathy score (T)...
  Getting attachment score (Y) [strict]...
  Getting attachment score (Y) [strict]...


NameError: name 'att_lenient' is not defined

In [ ]:
# Examine the scored results
print("Scored Subset Results (v2 - Contextual):")
print("="*70)
print(df_subset_scored[['turn_pair_id', 'model', 'empathy_score', 'attachment_score']])

print("\n" + "="*70)
print("Score Distribution:")
print(f"\nEmpathy Scores:")
print(df_subset_scored['empathy_score'].value_counts().sort_index())
print(f"\nAttachment Scores (Contextual):")
print(df_subset_scored['attachment_score'].value_counts().sort_index())

Scored Subset Results (v2):
                               turn_pair_id               model empathy_score  \
0  0d2231ad2232cb26b6b4fb75a19ddc93_1701994  gpt-3.5-turbo-0613             2   
1  350c2f00ead0f1414ef4edf68f352d9e_1697823  gpt-3.5-turbo-0613             2   
2   1f8e543b0a918d6334270ec1e6860943_532466  gpt-3.5-turbo-0301             2   
3  8e24f7f88936429c0ce951519990342b_2781437  gpt-4-0125-preview             1   
4  8e24f7f88936429c0ce951519990342b_2781442  gpt-4-0125-preview             1   

  attachment_score_strict attachment_score_lenient attachment_score  
0                       3                        6                5  
1                       3                        6                5  
2                       3                        6                5  
3                       3                        5                5  
4                       3                        6                5  

Score Distribution:

Empathy Scores:
empathy_score
1    2
2    3

## 4. Score Subset of 60 Conversations

Testing the contextual scoring on 60 turn pairs (instead of all 223).

**Note**: This makes ~120 API calls (60 × 2: empathy + attachment) and should take ~3-5 minutes.

In [ ]:
# Score a SUBSET of 60 turn pairs with contextual approach
TEST_SIZE = 60

print(f"Testing contextual scoring on {TEST_SIZE} turn pairs...")
print("This should take ~3-5 minutes (2 API calls per pair).")
print("="*70)

# Take first 60 conversations
df_test = df_sample.head(TEST_SIZE).copy()

print(f"\nScoring {len(df_test)} turn pairs...")
df_sample_scored_v2 = score_conversations(df_test, verbose=True)

Testing new binary-friendly prompts on 60 turn pairs...
This should take ~5-7 minutes (3 API calls per pair).

Scoring 60 turn pairs...
--- Processing turn pair 1/60 (ID: 0d2231ad2232cb26b6b4fb75a19ddc93_1701994) ---
  Getting empathy score (T)...
  Getting attachment score (Y) [strict]...
  Getting attachment score (Y) [lenient]...
  Getting attachment score (Y) [strict]...
  Getting attachment score (Y) [lenient]...
  Scores: Empathy=2, Attachment(strict)=3, Attachment(lenient)=6, Final=5

--- Processing turn pair 2/60 (ID: 350c2f00ead0f1414ef4edf68f352d9e_1697823) ---
  Getting empathy score (T)...
  Scores: Empathy=2, Attachment(strict)=3, Attachment(lenient)=6, Final=5

--- Processing turn pair 2/60 (ID: 350c2f00ead0f1414ef4edf68f352d9e_1697823) ---
  Getting empathy score (T)...
  Getting attachment score (Y) [strict]...
  Getting attachment score (Y) [strict]...
  Getting attachment score (Y) [lenient]...
  Scores: Empathy=2, Attachment(strict)=3, Attachment(lenient)=6, Final=5


In [ ]:
# Save the scored subset (v2 with contextual scoring)
output_path = '../data/scores/wildchat_60sample_scored_v2_contextual.csv'

# Create directory if it doesn't exist
os.makedirs('../data/scores', exist_ok=True)

print(f"Saving scored subset to: {output_path}")
df_sample_scored_v2.to_csv(output_path, index=False)

print(f"✓ Saved successfully!")
print(f"  Rows: {len(df_sample_scored_v2)}")
print(f"  Columns: {len(df_sample_scored_v2.columns)}")
print(f"  File size: {os.path.getsize(output_path) / (1024):.2f} KB")

## 5. Analyze v2 Score Distributions

In [ ]:
# Check for missing scores
print("Data Quality Check (v2):")
print("="*70)
print(f"Total turn pairs: {len(df_sample_scored_v2)}")
print(f"\nMissing empathy scores: {df_sample_scored_v2['empathy_score'].isna().sum()}")
print(f"Missing attachment (strict): {df_sample_scored_v2['attachment_score_strict'].isna().sum()}")
print(f"Missing attachment (lenient): {df_sample_scored_v2['attachment_score_lenient'].isna().sum()}")
print(f"Missing attachment (final): {df_sample_scored_v2['attachment_score'].isna().sum()}")

In [ ]:
# Score distributions (v2)
print("\nEmpathy Score Distribution (v2):")
print("="*70)
empathy_counts = df_sample_scored_v2['empathy_score'].value_counts().sort_index()
empathy_pct = (empathy_counts / len(df_sample_scored_v2) * 100).round(2)

print(f"{'Score':>6} {'Count':>10} {'Percentage':>12}")
print("-"*30)
for score in range(1, 8):
    count = empathy_counts.get(score, 0)
    pct = empathy_pct.get(score, 0.0)
    print(f"{score:>6} {count:>10} {pct:>11.2f}%")
print("-"*30)
print(f"{'Total':>6} {len(df_sample_scored_v2):>10} {'100.00%':>12}")

print(f"\nSummary Statistics:")
print(df_sample_scored_v2['empathy_score'].describe())

In [ ]:
# Attachment distribution - contextual scoring
print("\nAttachment Score Distribution (v2 - Contextual):")
print("="*70)

counts = df_sample_scored_v2['attachment_score'].value_counts().sort_index()
pct = (counts / len(df_sample_scored_v2) * 100).round(2)

print(f"{'Score':>6} {'Count':>10} {'Percentage':>12}")
print("-"*30)
for score in range(1, 8):
    count = counts.get(score, 0)
    p = pct.get(score, 0.0)
    print(f"{score:>6} {count:>10} {p:>11.2f}%")
print("-"*30)
print(f"{'Total':>6} {len(df_sample_scored_v2):>10} {'100.00%':>12}")

# High attachment (>=5)
high_att = df_sample_scored_v2['attachment_score'].ge(5).sum()
high_att_pct = (high_att / len(df_sample_scored_v2) * 100).round(2)
print(f"\nHigh Attachment (≥5): {high_att} ({high_att_pct}%)")

print(f"\nSummary Stats:")
print(f"  Mean: {df_sample_scored_v2['attachment_score'].mean():.2f}")
print(f"  Median: {df_sample_scored_v2['attachment_score'].median():.1f}")
print(f"  Std: {df_sample_scored_v2['attachment_score'].std():.2f}")

In [ ]:
# Visualize attachment distribution
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

ax.hist(df_sample_scored_v2['attachment_score'].dropna(), bins=np.arange(0.5, 8.5, 1),
        color='steelblue', edgecolor='black', alpha=0.7)
ax.set_title('Attachment Score Distribution (v2 - Contextual)', fontsize=14, fontweight='bold')
ax.set_xlabel('Score (1-7)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_xticks(range(1, 8))

mean_val = df_sample_scored_v2['attachment_score'].mean()
ax.axvline(mean_val, color='red', linestyle='--',
           label=f'Mean: {mean_val:.2f}')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Compare v1 vs v2

Load the original v1 scores and compare distributions

In [ ]:
# Load v1 scores if available
v1_path = '../data/scores/wildchat_sample_scored.csv'

if os.path.exists(v1_path):
    print(f"Loading v1 scores from: {v1_path}")
    df_v1 = pd.read_csv(v1_path)
    
    print("\nComparing v1 vs v2 Attachment Scores:")
    print("="*70)
    
    # High attachment comparison
    v1_high = df_v1['attachment_score'].ge(5).sum()
    v2_high = df_sample_scored_v2['attachment_score'].ge(5).sum()
    
    v1_high_pct = (v1_high / len(df_v1) * 100).round(2)
    v2_high_pct = (v2_high / len(df_sample_scored_v2) * 100).round(2)
    
    print(f"\nHigh Attachment (≥5):")
    print(f"  v1 (user reply only): {v1_high} / {len(df_v1)} ({v1_high_pct}%)")
    print(f"  v2 (contextual):      {v2_high} / {len(df_sample_scored_v2)} ({v2_high_pct}%)")
    print(f"  Improvement: {v2_high - v1_high} pairs (+{v2_high_pct - v1_high_pct:.2f}%)")
    
    # Mean comparison
    v1_mean = df_v1['attachment_score'].mean()
    v2_mean = df_sample_scored_v2['attachment_score'].mean()
    
    print(f"\nMean Attachment Score:")
    print(f"  v1: {v1_mean:.2f}")
    print(f"  v2: {v2_mean:.2f}")
    print(f"  Difference: {v2_mean - v1_mean:+.2f}")
    
else:
    print(f"v1 scores not found at {v1_path}")
    print("Run 02_pilot_tests.ipynb first to generate v1 scores for comparison.")

In [ ]:
# Side-by-side comparison plot (if v1 exists)
if os.path.exists(v1_path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # v1
    axes[0].hist(df_v1['attachment_score'].dropna(), bins=np.arange(0.5, 8.5, 1),
                 color='lightcoral', edgecolor='black', alpha=0.7)
    axes[0].set_title('v1 (User Reply Only)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Attachment Score', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)
    axes[0].set_xticks(range(1, 8))
    axes[0].axvline(df_v1['attachment_score'].mean(), color='red', linestyle='--',
                    label=f'Mean: {df_v1["attachment_score"].mean():.2f}')
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # v2
    axes[1].hist(df_sample_scored_v2['attachment_score'].dropna(), bins=np.arange(0.5, 8.5, 1),
                 color='steelblue', edgecolor='black', alpha=0.7)
    axes[1].set_title('v2 (Contextual)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Attachment Score', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].set_xticks(range(1, 8))
    axes[1].axvline(df_sample_scored_v2['attachment_score'].mean(), color='red', linestyle='--',
                    label=f'Mean: {df_sample_scored_v2["attachment_score"].mean():.2f}')
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.suptitle('Attachment Score Distribution: v1 vs v2', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

## 7. Summary

### Contextual Scoring Approach (v2)

**Main Goal**: Improve attachment score distribution by providing context to the judge

**Key Changes from v1**:
1. **Contextual evaluation**: Judge sees BOTH assistant response + user reply (not just user reply)
2. **Single-pass scoring**: Simplified from dual-pass aggregation to direct contextual evaluation
3. **Better signal detection**: Context helps identify attachment indicators more accurately

### Test Results (60 conversations)

Run the analysis cells above to see:
- Attachment score distribution
- High attachment (≥5) frequency
- Comparison to v1 (user reply only)

### Files Generated

- `data/scores/wildchat_60sample_scored_v2_contextual.csv`
  - Test of contextual scoring approach
  - Columns: `empathy_score`, `attachment_score`

### Next Steps

1. ✅ Test contextual scoring on 60 pairs
2. Evaluate distribution improvements vs v1
3. If successful, scale to full dataset
4. Generate embeddings for propensity score matching
5. Perform causal analysis

## 6.5 Distribution Analysis

Analyzing the score distribution from contextual scoring

In [ ]:
# DISTRIBUTION ANALYSIS: Contextual scoring results
print("="*70)
print("ATTACHMENT SCORE DISTRIBUTION ANALYSIS (V2 - Contextual)")
print("="*70)

# Calculate distribution metrics
score_counts = df_sample_scored_v2['attachment_score'].value_counts().sort_index()
total_valid = df_sample_scored_v2['attachment_score'].notna().sum()

print(f"\nScore Distribution:")
print(f"{'Score':>6} {'Count':>10} {'Percentage':>12}")
print("-"*30)
for score in range(1, 8):
    count = score_counts.get(score, 0)
    pct = (count / total_valid * 100) if total_valid > 0 else 0
    print(f"{score:>6} {count:>10} {pct:>11.2f}%")

# Calculate score=4 frequency
score_4_count = score_counts.get(4, 0)
score_4_pct = (score_4_count / total_valid * 100) if total_valid > 0 else 0

print(f"\n" + "="*70)
print("SCORE=4 FREQUENCY:")
print("="*70)
print(f"Score=4: {score_4_count} / {total_valid} ({score_4_pct:.1f}%)")

# Data split for causal analysis
control = (df_sample_scored_v2['attachment_score'] <= 3).sum()
excluded = score_4_count
treatment = (df_sample_scored_v2['attachment_score'] >= 5).sum()

print(f"\n" + "="*70)
print("DATA SPLIT FOR CAUSAL ANALYSIS:")
print("="*70)
print(f"Control (≤3):     {control:3d} ({control/total_valid*100:5.1f}%)")
print(f"EXCLUDED (=4):    {excluded:3d} ({excluded/total_valid*100:5.1f}%)")
print(f"Treatment (≥5):   {treatment:3d} ({treatment/total_valid*100:5.1f}%)")
print(f"Usable data:      {control + treatment:3d} ({(control + treatment)/total_valid*100:5.1f}%)")

# Summary statistics
print(f"\n" + "="*70)
print("SUMMARY STATISTICS:")
print("="*70)
print(f"Mean:   {df_sample_scored_v2['attachment_score'].mean():.2f}")
print(f"Median: {df_sample_scored_v2['attachment_score'].median():.1f}")
print(f"Std:    {df_sample_scored_v2['attachment_score'].std():.2f}")
print(f"Min:    {df_sample_scored_v2['attachment_score'].min():.0f}")
print(f"Max:    {df_sample_scored_v2['attachment_score'].max():.0f}")
print("="*70)